# 03 — Live data from a running simulation

`runScriptAsync` pauses the run every *N* steps and calls your async callback
with fresh data — step number, particle snapshot, and the current values of
any computes you name. This is what powers live visualization and plots.

> **One rule**: inside the callback, make all `lammps.*` calls **before** the
> first `await`. While the wasm run is suspended it cannot be re-entered after
> your callback yields.

In [ ]:
// Load lammps.js (served by this site under ./lammps/). Run this cell first.
// The site root is derived from wherever this code runs: the kernel iframe
// inherits the page URL ({site}/lab/…), the worker kernel lives under
// {site}/extensions/….
const base = globalThis.document?.baseURI ?? location.href;
globalThis.SITE ??= base.replace(/(extensions|lab|notebooks|files|tree|repl|consoles|edit)\/.*$/, "");
globalThis.LammpsClient ??= (await import(new URL("lammps/client.js", globalThis.SITE))).LammpsClient;
"lammps.js loaded ✓"

In [ ]:
// A tiny console "plot": unicode sparkline of an array of numbers.
globalThis.spark = (xs) => {
  const min = Math.min(...xs), max = Math.max(...xs), glyphs = "▁▂▃▄▅▆▇█";
  return xs.map((v) => glyphs[Math.min(7, Math.floor(((v - min) / ((max - min) || 1)) * 8))]).join("");
};
"spark() defined ✓"

## Watch the temperature equilibrate

The NVT thermostat drags the system from T = 3.0 down to T = 0.7. The callback
records the `ctemp` compute every 50 steps.

In [ ]:
globalThis.lammps = await LammpsClient.create({ print: (line) => console.log(line) });
lammps.start();

globalThis.temps = [];
await lammps.runScriptAsync(
  `
  units         lj
  timestep      0.005
  atom_style    atomic
  lattice       fcc 0.8442
  region        box block 0 3 0 3 0 3
  create_box    1 box
  create_atoms  1 box
  mass          1 1.0
  velocity      all create 3.0 87287
  pair_style    lj/cut 2.5
  pair_coeff    1 1 1.0 1.0 2.5
  compute       ctemp all temp
  compute       cke all ke
  fix           1 all nvt temp 3.0 0.7 0.5
  thermo        2000
  run           6000
  `,
  async (data) => {
    // lammps.* calls and data reads happen here, before any await.
    const { ctemp, cke } = data.computeScalars;
    temps.push(ctemp);
    if (data.step % 1000 === 0) {
      console.log(`step ${String(data.step).padStart(5)}   T = ${ctemp.toFixed(3)}   KE = ${cke.toFixed(1)}`);
    }
  },
  { every: 50, computeScalars: ["ctemp", "cke"] }
);
console.log("callbacks received:", temps.length);

## "Plot" it

A proper plotting helper is on the roadmap — until then, a sparkline shows the
quench nicely.

In [ ]:
console.log(`T  ${temps[0].toFixed(2)} ${spark(temps)} ${temps.at(-1).toFixed(2)}`);
lammps.dispose();